#Experiment-6: GloVe Word Embeddings and Language Modelling


#### To implement the GloVe (Global Vectors for Word Representation) algorithm to generate word embeddings from the text corpus and find most similar words of “King”.

#####1. Without using any pretrained model

In [3]:
!pip install numpy scipy nltk

In [5]:
import re
import numpy as np
import torch
import torch.nn as nn
from collections import Counter, defaultdict

with open("glove_corpus.txt", encoding="utf-8") as f:
    text = f.read().lower()
tokens = re.findall(r"[a-z]+", text)

counts = Counter(tokens)
vocab = list(counts.keys())
word2id = {w: i for i, w in enumerate(vocab)}
id2word = {i: w for w, i in word2id.items()}
V = len(vocab)
ids = [word2id[w] for w in tokens]
print(f"Tokens: {len(tokens)}, Vocab size: {V}")

WINDOW = 5
cooc = defaultdict(float)
for i, center in enumerate(ids):
    for j in range(max(0, i - WINDOW), i):
        d = i - j
        cooc[(center, ids[j])] += 1.0 / d
        cooc[(ids[j], center)] += 1.0 / d

pairs = list(cooc.keys())
i_idx = torch.tensor([p[0] for p in pairs])
j_idx = torch.tensor([p[1] for p in pairs])
X = torch.tensor([cooc[p] for p in pairs], dtype=torch.float32)
print("Non-zero co-occurrences:", len(pairs))

class GloVe(nn.Module):
    def __init__(self, V, dim):
        super().__init__()
        self.w  = nn.Embedding(V, dim)   # word vectors
        self.wt = nn.Embedding(V, dim)   # context vectors
        self.b  = nn.Embedding(V, 1)
        self.bt = nn.Embedding(V, 1)
        for e in (self.w, self.wt):
            nn.init.uniform_(e.weight, -0.5 / dim, 0.5 / dim)
        for e in (self.b, self.bt):
            nn.init.zeros_(e.weight)

    def forward(self, i, j):
        return (self.w(i) * self.wt(j)).sum(1) + self.b(i).squeeze() + self.bt(j).squeeze()

DIM, XMAX, ALPHA = 50, 100.0, 0.75
EPOCHS, BATCH, LR = 100, 1024, 0.05

model = GloVe(V, DIM)
opt = torch.optim.Adagrad(model.parameters(), lr=LR)

for epoch in range(EPOCHS):
    perm = torch.randperm(len(X))
    total = 0.0
    for s in range(0, len(X), BATCH):
        b = perm[s:s + BATCH]
        xi, xj, x = i_idx[b], j_idx[b], X[b]
        weight = torch.clamp((x / XMAX) ** ALPHA, max=1.0)
        loss = (weight * (model(xi, xj) - torch.log(x)) ** 2).sum()
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS}  loss: {total:.4f}")

embeddings = (model.w.weight + model.wt.weight).detach().numpy()
embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True) + 1e-9

def most_similar(word, topn=10):
    if word not in word2id:
        print(f"'{word}' not in corpus vocabulary")
        return
    sims = embeddings @ embeddings[word2id[word]]
    for k in np.argsort(-sims)[1:topn + 1]:
        print(f"{id2word[k]:15s} {sims[k]:.4f}")

print("\nMost similar to 'king':")
most_similar("king")

Tokens: 148, Vocab size: 68
Non-zero co-occurrences: 779
Epoch 10/100  loss: 1.0165
Epoch 20/100  loss: 0.1507
Epoch 30/100  loss: 0.0433
Epoch 40/100  loss: 0.0147
Epoch 50/100  loss: 0.0052
Epoch 60/100  loss: 0.0019
Epoch 70/100  loss: 0.0007
Epoch 80/100  loss: 0.0003
Epoch 90/100  loss: 0.0001
Epoch 100/100  loss: 0.0001

Most similar to 'king':
the             0.7631
kingdom         0.6860
queen           0.6637
is              0.6330
dog             0.6068
rules           0.5674
cat             0.5588
and             0.4873
princess        0.4040
of              0.3536


#####2. Using Pretrained model (Gensim)

In [7]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 37.6 MB/s  0:00:00


In [8]:
import gensim.downloader as api

glove = api.load("glove-wiki-gigaword-100")

print("Most similar to 'king':")
for word, score in glove.most_similar("king", topn=10):
    print(f"{word:15s} {score:.4f}")

print("\nking - man + woman =")
for word, score in glove.most_similar(positive=["king", "woman"], negative=["man"], topn=5):
    print(f"{word:15s} {score:.4f}")

[==================================================] 100.0% 128.1/128.1MB downloaded
Most similar to 'king':
prince          0.7682
queen           0.7508
son             0.7021
brother         0.6986
monarch         0.6978
throne          0.6920
kingdom         0.6811
father          0.6802
emperor         0.6713
ii              0.6676

king - man + woman =
queen           0.7699
monarch         0.6843
throne          0.6756
daughter        0.6595
princess        0.6521


####To implement a statistical language model from scratch using the N-gram approach to predict the next word in a sequence based on the preceding words in a text corpus (shown below).

natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.

In [9]:
import re
import random
from collections import defaultdict, Counter

corpus = """
natural language processing is a branch of artificial intelligence.
natural language processing enables computers to understand human language.
machine learning is a branch of artificial intelligence.
deep learning is a part of machine learning.
language models learn patterns from text data.
a language model predicts the next word in a sentence.
natural language processing uses language models.
artificial intelligence includes machine learning and deep learning.
"""
sentences = [s for s in corpus.strip().split("\n") if s.strip()]

def tokenize(s):
    return re.findall(r"[a-z]+", s.lower())

class NGramLM:
    def __init__(self, n=3):
        self.n = n
        self.counts = {k: defaultdict(Counter) for k in range(n)}

    def train(self, sentences):
        for s in sentences:
            toks = ["<s>"] * (self.n - 1) + tokenize(s) + ["</s>"]
            for i in range(self.n - 1, len(toks)):
                for k in range(self.n):            # context lengths 0..n-1
                    ctx = tuple(toks[i - k:i]) if k > 0 else ()
                    self.counts[k][ctx][toks[i]] += 1

    def next_word_probs(self, context):
        words = ["<s>"] * (self.n - 1) + tokenize(context)
        for k in range(self.n - 1, -1, -1):        # backoff: long -> short context
            ctx = tuple(words[-k:]) if k > 0 else ()
            if ctx in self.counts[k]:
                c = self.counts[k][ctx]
                total = sum(c.values())
                return {w: v / total for w, v in c.items()}, k
        return {}, 0

    def predict(self, context, topk=3):
        probs, used = self.next_word_probs(context)
        ranked = sorted(probs.items(), key=lambda x: -x[1])[:topk]
        return ranked, used

    def generate(self, context, max_words=10):
        out = tokenize(context)
        for _ in range(max_words):
            probs, _ = self.next_word_probs(" ".join(out))
            words, p = zip(*probs.items())
            nxt = random.choices(words, weights=p)[0]
            if nxt == "</s>":
                break
            out.append(nxt)
        return " ".join(out)

model = NGramLM(n=3)          # n=2 for bigram
model.train(sentences)

tests = ["natural language", "machine", "deep learning is", "language models",
         "artificial", "a language"]
for t in tests:
    preds, used = model.predict(t)
    print(f"\nInput: '{t}'  (context length used: {used})")
    for w, p in preds:
        print(f"   {w:15s} P = {p:.3f}")

print("\nGenerated:", model.generate("natural language"))


Input: 'natural language'  (context length used: 2)
   processing      P = 1.000

Input: 'machine'  (context length used: 2)
   learning        P = 1.000

Input: 'deep learning is'  (context length used: 2)
   a               P = 1.000

Input: 'language models'  (context length used: 2)
   learn           P = 0.500
   </s>            P = 0.500

Input: 'artificial'  (context length used: 2)
   intelligence    P = 1.000

Input: 'a language'  (context length used: 2)
   model           P = 1.000

Generated: natural language processing is a branch of artificial intelligence includes machine learning
